# LlamaIndex Agentic RAG (SEC v2 Parity)

This notebook ports the LangGraph SEC v2 state machine to a LlamaIndex event-driven Workflow.

Parity goals:
- Same retrieval stack: BM25 + Dense (Chroma) + RRF + CrossEncoder rerank.
- Same control flow: planner -> retriever -> context evaluator -> generator -> critic -> repair/retry.
- Same model strings and temperatures as the LangGraph notebook.
- No re-embedding: connect to existing Chroma collection via ChromaVectorStore.

In [1]:
# Install/repair dependencies in the active notebook kernel if needed.
# %pip install -U chromadb python-dotenv rank-bm25 sentence-transformers pydantic
# %pip install -U llama-index-core llama-index-vector-stores-chroma
# %pip install -U llama-index-llms-groq llama-index-llms-gemini

In [2]:
import os
import sys
import re
import json
import time
import threading
from dataclasses import dataclass
from pathlib import Path
from collections import deque
from typing import Any, Dict, List, Optional, TypedDict, Literal

import numpy as np
import pandas as pd
import chromadb

from dotenv import load_dotenv
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from pydantic import BaseModel, Field, field_validator, model_validator

from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core.workflow import Workflow, Event, StartEvent, StopEvent, step, Context

from llama_index.llms.groq import Groq as LIGroq
from llama_index.llms.gemini import Gemini as LIGemini

# ========== IMPORT FROM SHARED MODULES ==========
# Resolve repo root robustly from notebook location.
CWD = Path.cwd()

def detect_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "config.py").exists() and (p / "shared_retriever.py").exists():
            return p
    for p in [start, *start.parents]:
        if (p / ".env").exists():
            return p
    return start

PROJECT_ROOT = detect_project_root(CWD)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Path helper used by drift/ingestion cells

def resolve_path_from_env(path_str: str) -> Path:
    p = Path(path_str)
    if p.exists():
        return p
    if not p.is_absolute():
        c1 = (CWD / p).resolve()
        if c1.exists():
            return c1
        c2 = (PROJECT_ROOT / p).resolve()
        if c2.exists():
            return c2
    return p

# Load .env and import shared modules
load_dotenv(PROJECT_ROOT / ".env")

from config import CONFIG, resolve_model_name, resolve_fallback_model_names, get_provider_order
from shared_retriever import initialize_corpus, CorpusIndex

print(f"Ã¢Å“â€œ Imported CONFIG from {PROJECT_ROOT / 'config.py'}")
print(f"Ã¢Å“â€œ Imported shared_retriever from {PROJECT_ROOT / 'shared_retriever.py'}")

W0321 22:55:35.930000 4636 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


c:\Users\wenxu\miniconda3\Lib\site-packages\llama_index\llms\gemini\base.py:21: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


Loading embedding model: sentence-transformers/all-mpnet-base-v2...


Loading reranker model: cross-encoder/ms-marco-MiniLM-L-6-v2...


Ã¢Å“â€œ Imported CONFIG from C:\Users\wenxu\GenAI-with-LLMs\config.py
Ã¢Å“â€œ Imported shared_retriever from C:\Users\wenxu\GenAI-with-LLMs\shared_retriever.py


In [3]:
# ========== ADAPTER: Bridge shared_retriever output to LlamaIndex-compatible formats ==========
# shared_retriever returns Dict; LlamaIndex may expect RetrievedChunk or NodeWithScore
# Define adapter class for compatibility with LlamaIndex workflow

@dataclass
class RetrievedChunk:
    """Adapter dataclass matching shared_retriever output schema."""
    doc_name: str
    company: str
    page_num: int
    chunk_id: str
    raw_chunk: str
    contextual_chunk: str
    score: float
    source: str
    ticker: Optional[str] = None


def dict_to_retrieved_chunk(d: Dict[str, Any]) -> RetrievedChunk:
    """Convert shared_retriever Dict output to RetrievedChunk adapter."""
    meta = d.get('metadata', {})
    return RetrievedChunk(
        doc_name=meta.get('doc_name', ''),
        company=meta.get('company', ''),
        page_num=meta.get('page_num', 0),
        chunk_id=d.get('chunk_id', ''),
        raw_chunk=d.get('raw_text', ''),
        contextual_chunk=d.get('text', ''),
        score=float(d.get('score', 0.0)),
        source=d.get('source', ''),
        ticker=meta.get('ticker', None),
    )


def extract_retrieved_doc_names(chunks: List[RetrievedChunk]) -> List[str]:
    return list(dict.fromkeys([c.doc_name for c in chunks]))


def cleanup_planner_query(
    query: Optional[str],
    ticker: Optional[str] = None,
    filing_year: Optional[int] = None,
    form_type: Optional[str] = None,
) -> str:
    """Clean and augment planner query with metadata hints."""
    cleaned = (query or "").strip()
    if not cleaned:
        cleaned = "financial metric from SEC filing"
    upper = cleaned.upper()
    if upper.startswith("SELECT ") and " FROM " in upper:
        cleaned = cleaned.replace("SELECT", "").replace("FROM filings", "").strip()
    for token in ("WHERE", "FROM", "AND", "=", "'", '"'):
        cleaned = cleaned.replace(token, " ")
    cleaned = cleaned.replace("filing year", "filing_year")
    cleaned = cleaned.replace("|", " ")
    cleaned = cleaned.replace("_", " ")
    cleaned = cleaned.replace("company", "")
    cleaned = cleaned.replace("ticker", "")
    cleaned = " ".join(cleaned.split())

    parts = [cleaned]
    if ticker and ticker not in cleaned:
        parts.insert(0, ticker)
    if filing_year and str(filing_year) not in cleaned:
        parts.append(str(filing_year))
    if form_type and form_type not in cleaned:
        parts.append(form_type)
    return " ".join([p for p in parts if p]).strip()


def fails_retrieval_sanity_check(
    question: str, 
    sub_queries: List[Dict[str, Any]], 
    retrieved: List[RetrievedChunk]
) -> bool:
    """Check if retrieved chunks match expected companies/periods from planner."""
    if not question.strip():
        return False
    if not CONFIG["enable_retrieval_sanity_check"]:
        return False
    if not retrieved:
        return False
    expected_tickers = {sq.get("ticker") for sq in sub_queries if sq.get("ticker")}
    if not expected_tickers:
        return False
    present_tickers = {c.ticker for c in retrieved if c.ticker}
    if not present_tickers:
        return False
    return not expected_tickers.issubset(present_tickers)


# ========== INITIALIZE SHARED CORPUS ==========
print("Initializing shared CorpusIndex from shared_retriever.py...")
global_index = initialize_corpus(
    chunks_jsonl=CONFIG["sec_chunks_path"],
    chroma_db_path=CONFIG["chroma_db_path"],
)
print(f"Ã¢Å“â€œ Corpus initialized: {global_index.df.shape[0]:,} chunks, "
      f"{global_index.chroma_collection.count():,} vectors")
print(f"Ã¢Å“â€œ Adjacent chunk expansion: ENABLED")
print(f"Ã¢Å“â€œ Dense model: {CONFIG['dense_model_name']}")


Initializing shared CorpusIndex from shared_retriever.py...
Loading chunks from C:\Users\wenxu\GenAI-with-LLMs\sec_rag_team_share\sec_data\chunks\sec_chunks.jsonl...


Loaded 13275 chunks
Building BM25 index...


Connecting to Chroma DB at C:\Users\wenxu\GenAI-with-LLMs\sec_rag_team_share\chroma_db...


Chroma collection: 12725 vectors


CorpusIndex ready.
Ã¢Å“â€œ Corpus initialized: 13,275 chunks, 12,725 vectors
Ã¢Å“â€œ Adjacent chunk expansion: ENABLED
Ã¢Å“â€œ Dense model: sentence-transformers/all-mpnet-base-v2


In [4]:
# ========== RECOVERY: Reinitialize corpus if needed ==========
print("[Recovery] Reinitializing shared CorpusIndex...")
global_index = initialize_corpus(
    chunks_jsonl=CONFIG["sec_chunks_path"],
    chroma_db_path=CONFIG["chroma_db_path"],
)
print(f"[Recovery] Ã¢Å“â€œ Corpus ready: {global_index.df.shape[0]:,} chunks")


[Recovery] Reinitializing shared CorpusIndex...
Loading chunks from C:\Users\wenxu\GenAI-with-LLMs\sec_rag_team_share\sec_data\chunks\sec_chunks.jsonl...


Loaded 13275 chunks
Building BM25 index...


Connecting to Chroma DB at C:\Users\wenxu\GenAI-with-LLMs\sec_rag_team_share\chroma_db...
Chroma collection: 12725 vectors


CorpusIndex ready.


[Recovery] Ã¢Å“â€œ Corpus ready: 13,275 chunks


In [5]:
class SubQuery(BaseModel):
    query: Optional[str] = Field(default=None, description="Search-optimized sub-query text for retrieval")
    ticker: Optional[str] = Field(default=None, description="Company ticker if company-specific")
    filing_year: Optional[int] = Field(default=None, description="Calendar year when the filing was submitted")
    form_type: Optional[str] = Field(default=None, description="10-K or 10-Q if specified")

    @field_validator("query", mode="before")
    @classmethod
    def normalize_query(cls, v):
        if isinstance(v, dict):
            parts: List[str] = []
            for key in ("query", "query_field", "metric", "line_item", "table", "topic"):
                value = v.get(key)
                if value is not None and str(value).strip():
                    parts.append(str(value).strip())
            fields = v.get("fields")
            if isinstance(fields, list) and fields:
                parts.append(" ".join(str(item).strip() for item in fields if str(item).strip()))
            return " | ".join(dict.fromkeys(parts)) if parts else None
        return v

    @field_validator("ticker", mode="before")
    @classmethod
    def normalize_ticker(cls, v):
        if v is None:
            return None
        cleaned = str(v).strip().upper()
        return cleaned or None

    @field_validator("form_type", mode="before")
    @classmethod
    def normalize_form_type(cls, v):
        if v is None:
            return None
        cleaned = str(v).strip().upper()
        return cleaned if cleaned in {"10-K", "10-Q"} else None


class PlannerOutput(BaseModel):
    needs_decomposition: bool = Field(
        description="True when the question requires multiple distinct filings, periods, or companies."
    )
    sub_queries: List[SubQuery] = Field(
        description="One sub-query for single-filing questions, otherwise 2-3 sub-queries split by company or period."
    )

    @model_validator(mode="before")
    @classmethod
    def normalize_keys(cls, data):
        if not isinstance(data, dict):
            return data
        normalized = dict(data)
        if "sub_queries" not in normalized:
            for alias in ("queries", "sub_query", "query"):
                if alias in normalized:
                    value = normalized[alias]
                    normalized["sub_queries"] = value if isinstance(value, list) else [value]
                    break
        if "needs_decomposition" not in normalized:
            sub_queries = normalized.get("sub_queries", [])
            normalized["needs_decomposition"] = isinstance(sub_queries, list) and len(sub_queries) > 1
        return normalized

    @field_validator("needs_decomposition", mode="before")
    @classmethod
    def coerce_bool(cls, v):
        if isinstance(v, str):
            return v.strip().lower() in {"true", "1", "yes"}
        return bool(v)


class ContextEvalOutput(BaseModel):
    relevant: bool
    reason: str


class CriticOutput(BaseModel):
    decision: Literal["accept", "repair", "insufficient_data"]
    reason: str


class RepairOutput(BaseModel):
    decision: Literal["accept", "warn", "refuse"]
    revised_answer: str
    needs_new_retrieval: bool
    reason: str


class JudgeOutput(BaseModel):
    score: float
    claims_covered: float
    reason: str


PLANNER_SYSTEM = (
    "You are a financial research planner working with SEC filings from these companies:\n"
    "AAPL (Apple), BAC (Bank of America), BRK (Berkshire Hathaway), COST (Costco), "
    "CVX (Chevron), JNJ (Johnson & Johnson), JPM (JPMorgan Chase), MSFT (Microsoft), "
    "NVDA (NVIDIA), UNH (UnitedHealth), WMT (Walmart), XOM (ExxonMobil).\n\n"
    "Decide whether the question requires data from multiple distinct filings, then produce sub-queries.\n\n"
    "Decompose (needs_decomposition=True) when the question:\n"
    "  Ã¢â‚¬Â¢ Explicitly compares two different fiscal years (e.g. 2023 vs 2024)\n"
    "  Ã¢â‚¬Â¢ Compares two different companies\n\n"
    "Do NOT decompose single-period, single-company questions.\n\n"
    "For each sub-query set ticker if company-specific, filing_year if year-specific.\n"
    "Every sub-query object must include a non-empty query field.\n"
    "filing_year = the calendar year the 10-K or 10-Q was filed "
    "(e.g. Apple FY2024 10-K was filed in November 2024, so filing_year=2024). "
    "Return a valid JSON object only that matches the requested schema."
)

CONTEXT_EVAL_SYSTEM = (
    "You judge whether retrieved context is relevant and sufficient to answer a financial question. "
    "Mark as not relevant only if the context is clearly from the wrong company or period, or completely empty. "
    "Partial tables and incomplete passages still count as relevant if they contain the right metric. "
    "Return valid JSON only with keys relevant and reason."
)

GENERATOR_SYSTEM = (
    "You are a financial analyst answering questions using only the retrieved filing context. "
    "Be precise with numbers, units, fiscal periods, and line-item names. "
    "If the context does not contain the answer, say so explicitly rather than estimating."
)

CRITIC_SYSTEM = (
    "You are a critic for a financial RAG system. Evaluate the answer and choose one decision: "
    "accept, repair, or insufficient_data. Use repair when the data is present but the answer used it incorrectly. "
    "Use insufficient_data only when the required financial data is absent from the retrieved context. "
    "Return valid JSON only with keys decision and reason."
)

REPAIR_SYSTEM = (
    "You repair a financial RAG answer after critique. Default to accept with a revised answer whenever possible. "
    "Pay close attention to fiscal year, quarter, units, sign, and line-item name. "
    "Set needs_new_retrieval=true only if critical data is completely absent from context. "
    "Return valid JSON only with keys decision, revised_answer, needs_new_retrieval, and reason."
)

JUDGE_CORRECTNESS_SYSTEM = (
    "Score the candidate answer against the reference answer from 0 to 1 for a financial QA task. "
    "1 = correct value, correct units, correct period. "
    "0 = wrong number, wrong company, wrong period, or completely missing. "
    "Give partial credit when the answer is directionally right but incomplete. "
    "Also set claims_covered: fraction of distinct facts/numbers in the reference present in the candidate. "
    "Return valid JSON only with keys score, claims_covered, and reason."
)


def _strip_code_fence(text: str) -> str:
    t = (text or "").strip()
    if t.startswith("```"):
        t = re.sub(r"^```[a-zA-Z0-9_\-]*", "", t).strip()
        if t.endswith("```"):
            t = t[:-3].strip()
    return t


def llm_complete(llm, prompt: str) -> str:
    for attempt in range(CONFIG["llm_max_retries"]):
        try:
            rate_limiter.wait()
            return llm.complete(prompt).text.strip()
        except Exception:
            if attempt == CONFIG["llm_max_retries"] - 1:
                raise
            time.sleep(CONFIG["llm_retry_base_delay_sec"] * (2 ** attempt))
    raise RuntimeError("Unreachable")


def invoke_structured(llm, schema, system: str, user: str, fallback):
    prompt = (
        f"System:\n{system}\n\n"
        "Return ONLY valid JSON, with no markdown fences.\n\n"
        f"User:\n{user}"
    )
    for attempt in range(CONFIG["llm_max_retries"]):
        try:
            raw = llm_complete(llm, prompt)
            data = json.loads(_strip_code_fence(raw))
            return schema.model_validate(data)
        except Exception:
            if attempt == CONFIG["llm_max_retries"] - 1:
                return fallback
            time.sleep(CONFIG["llm_retry_base_delay_sec"] * (2 ** attempt))
    return fallback


def invoke_text(llm, system: str, user: str, fallback_text: str = "") -> str:
    prompt = f"System:\n{system}\n\nUser:\n{user}"
    try:
        return llm_complete(llm, prompt)
    except Exception:
        return fallback_text


def llm_judge_correctness(question: str, reference_answer: str, candidate_answer: str) -> tuple[float, float, str]:
    judged = invoke_structured(
        judge_llm,
        JudgeOutput,
        JUDGE_CORRECTNESS_SYSTEM,
        (
            f"Question:\n{question}\n\n"
            f"Reference answer:\n{reference_answer}\n\n"
            f"Candidate answer:\n{candidate_answer}"
        ),
        JudgeOutput(score=0.0, claims_covered=0.0, reason="Judge fallback"),
    )
    return (
        max(0.0, min(1.0, float(judged.score))),
        max(0.0, min(1.0, float(judged.claims_covered))),
        judged.reason,
    )


def format_retrieved_context(chunks: List[RetrievedChunk], max_chunks: int, max_chars: int) -> str:
    selected = chunks[:max_chunks]
    parts: List[str] = []
    for i, c in enumerate(selected, start=1):
        parts.append(
            f"[Doc {i}] Company: {c.company} | Filing: {c.doc_name}\n"
            f"Content: {c.raw_chunk}"
        )
    joined = "\n\n".join(parts)
    return joined[:max_chars]

In [6]:
class AgentStateRequired(TypedDict):
    question: str
    # NOTE: global_index is now global, not per-state


class AgentState(AgentStateRequired, total=False):
    rewritten_query: str
    sub_queries: List[Dict[str, Any]]
    needs_decomposition: bool

    retrieved_chunks: List[Any]
    retrieved_doc_names: List[str]
    retrieval_sanity_failed: bool

    context_retries: int
    context_eval_relevant: bool

    generated_answer: str

    critic_decision: str
    critic_reason: str

    repair_used: bool
    repair_decision: str
    repair_reason: str
    needs_new_retrieval: bool
    repair_retrieval_count: int
    is_repair_retrieval: bool

    final_answer: str
    golden_answer: Optional[str]
    judge_score: Optional[float]
    judge_claims_covered: Optional[float]
    judge_reason: Optional[str]


class PlanEvent(Event):
    state: AgentState


class RetrieveEvent(Event):
    state: AgentState


class RetrievedEvent(Event):
    state: AgentState


class GenerateEvent(Event):
    state: AgentState


class CritiqueEvent(Event):
    state: AgentState


class RepairEvent(Event):
    state: AgentState


class RepairRetrieveEvent(Event):
    state: AgentState


class SecAgenticRAGWorkflow(Workflow):
    @step
    async def query_planning(self, _ctx: Context, ev: StartEvent) -> PlanEvent:
        question = ev.get("question")

        if not question:
            raise ValueError("Workflow requires question")

        state: AgentState = {
            "question": question,
            "context_retries": 0,
            "repair_used": False,
            "repair_retrieval_count": 0,
            "is_repair_retrieval": False,
            "retrieval_sanity_failed": False,
            "sub_queries": [],
            "needs_decomposition": False,
        }

        result = invoke_structured(
            planner_llm,
            PlannerOutput,
            PLANNER_SYSTEM,
            f"Question:\n{question}",
            PlannerOutput(needs_decomposition=False, sub_queries=[SubQuery(query=question)]),
        )

        sub_queries: List[Dict[str, Any]] = []
        for sq in result.sub_queries:
            sq_dict = sq.model_dump()
            if not sq_dict.get("query"):
                parts = [question]
                if sq_dict.get("ticker"):
                    parts.append(f"ticker {sq_dict['ticker']}")
                if sq_dict.get("filing_year"):
                    parts.append(f"filing year {sq_dict['filing_year']}")
                if sq_dict.get("form_type"):
                    parts.append(f"form {sq_dict['form_type']}")
                sq_dict["query"] = " | ".join(parts)
            sq_dict["query"] = cleanup_planner_query(
                sq_dict.get("query", ""),
                ticker=sq_dict.get("ticker"),
                filing_year=sq_dict.get("filing_year"),
                form_type=sq_dict.get("form_type"),
            )
            sub_queries.append(sq_dict)

        if not sub_queries:
            sub_queries = [{
                "query": cleanup_planner_query(question),
                "ticker": None,
                "filing_year": None,
                "form_type": None,
            }]

        state["sub_queries"] = sub_queries
        state["rewritten_query"] = sub_queries[0]["query"]
        state["needs_decomposition"] = result.needs_decomposition

        print(
            f"  [Planner] {'decomposed' if state['needs_decomposition'] else 'single'} | "
            f"{len(sub_queries)} sub-quer{'ies' if len(sub_queries) > 1 else 'y'}"
        )
        if state["needs_decomposition"]:
            for sq in sub_queries:
                print(
                    f"    -> {sq['query']}  "
                    f"(ticker={sq.get('ticker')}, year={sq.get('filing_year')}, form={sq.get('form_type')})"
                )

        # Event mapping: PlanEvent == LangGraph transition query_planner -> hybrid_retriever.
        return PlanEvent(state=state)

    @step
    async def hybrid_retriever(
        self,
        _ctx: Context,
        ev: PlanEvent | RetrieveEvent | RepairRetrieveEvent,
    ) -> RetrievedEvent:
        state = ev.state
        sub_queries = state.get("sub_queries", [])

        if isinstance(ev, RepairRetrieveEvent):
            state["is_repair_retrieval"] = True

        if len(sub_queries) <= 1:
            q = state.get("rewritten_query", state["question"])
            sq = sub_queries[0] if sub_queries else {}
            # ========== CALL SHARED RETRIEVER ==========
            retrieved_dicts = global_index.hybrid_search(
                query=q,
                top_k=CONFIG["rerank_top_k"],
                ticker=sq.get("ticker"),
                filing_year=sq.get("filing_year"),
                form_type=sq.get("form_type"),
            )
            # Convert shared_retriever Dict output to RetrievedChunk adapter
            retrieved = [dict_to_retrieved_chunk(d) for d in retrieved_dicts]
        else:
            per_k = CONFIG["decomposition_top_k_per_subquery"]
            merged: Dict[str, RetrievedChunk] = {}
            for sq in sub_queries:
                # ========== CALL SHARED RETRIEVER FOR EACH SUBQUERY ==========
                retrieved_dicts = global_index.hybrid_search(
                    query=sq["query"],
                    top_k=per_k,
                    ticker=sq.get("ticker"),
                    filing_year=sq.get("filing_year"),
                    form_type=sq.get("form_type"),
                )
                chunks = [dict_to_retrieved_chunk(d) for d in retrieved_dicts]
                for chunk in chunks:
                    if chunk.chunk_id not in merged or chunk.score > merged[chunk.chunk_id].score:
                        merged[chunk.chunk_id] = chunk
            retrieved = sorted(merged.values(), key=lambda x: x.score, reverse=True)[: CONFIG["rerank_top_k"]]

        normalized_retrieved = []
        for c in retrieved:
            if isinstance(c, RetrievedChunk):
                normalized_retrieved.append(c)
                continue
            normalized_retrieved.append(
                RetrievedChunk(
                    doc_name=getattr(c, "doc_name", ""),
                    company=getattr(c, "company", ""),
                    ticker=getattr(c, "ticker", ""),
                    form_type=getattr(c, "form_type", ""),
                    filing_year=int(getattr(c, "filing_year", 0) or 0),
                    page_num=int(getattr(c, "page_num", 0) or 0),
                    chunk_id=str(getattr(c, "chunk_id", "")),
                    raw_chunk=getattr(c, "raw_chunk", ""),
                    contextual_chunk=getattr(c, "contextual_chunk", ""),
                    score=float(getattr(c, "score", 0.0) or 0.0),
                    source=getattr(c, "source", "hybrid"),
                )
            )

        state["retrieved_chunks"] = normalized_retrieved
        state["retrieved_doc_names"] = extract_retrieved_doc_names(retrieved)
        state["retrieval_sanity_failed"] = fails_retrieval_sanity_check(
            state["question"],
            sub_queries,
            retrieved,
        )
        
        print(f"    [Retriever] Retrieved {len(retrieved)} chunks | Adjacent expansion: ACTIVE")

        # Event mapping: RetrievedEvent == LangGraph post-retrieval state before context_evaluator.
        return RetrievedEvent(state=state)

In [7]:
# ---------------------------------------------------------------------------
# SEC v3 parity patch for LlamaIndex Workflow
# Adds: drift detection + scrape-on-demand + ingestion retry branch
# ---------------------------------------------------------------------------

import requests
from datetime import datetime, timezone
from collections import defaultdict
from typing import Tuple


TICKER_TO_CIK = {
    "AAPL": "0000320193", "MSFT": "0000789019", "NVDA": "0001045810",
    "JPM": "0000019617", "BAC": "0000070858", "BRK-B": "0001067983",
    "JNJ": "0000200406", "UNH": "0000731766", "XOM": "0000034088",
    "CVX": "0000093410", "WMT": "0000104169", "COST": "0000909832",
}

COMPANY_NAMES_MAP = {
    "AAPL": "Apple Inc.", "MSFT": "Microsoft Corporation", "NVDA": "NVIDIA Corporation",
    "JPM": "JPMorgan Chase & Co.", "BAC": "Bank of America Corporation", "BRK-B": "Berkshire Hathaway Inc.",
    "JNJ": "Johnson & Johnson", "UNH": "UnitedHealth Group Incorporated", "XOM": "Exxon Mobil Corporation",
    "CVX": "Chevron Corporation", "WMT": "Walmart Inc.", "COST": "Costco Wholesale Corporation",
}

# SEC requires a descriptive User-Agent header for EDGAR access.
EDGAR_HEADERS = {"User-Agent": "GenAI-RAG-Project research@example.com"}


class DriftTracker:
    """Tracks insufficient-data misses and schedules scrape targets."""

    TRIGGER_THRESHOLD = 2

    def __init__(self, persist_path: Path):
        self.persist_path = persist_path
        self._counts: Dict[Tuple[str, str, str], int] = defaultdict(int)
        self._already_ingested: set[Tuple[str, str, str]] = set()
        self._load_existing()

    def _load_existing(self) -> None:
        if not self.persist_path.exists():
            return
        with self.persist_path.open("r", encoding="utf-8") as f:
            for line in f:
                try:
                    entry = json.loads(line)
                except Exception:
                    continue
                key = (
                    str(entry.get("ticker") or ""),
                    str(entry.get("filing_year") or ""),
                    str(entry.get("form_type") or "10-K"),
                )
                if entry.get("status") == "ingested":
                    self._already_ingested.add(key)
                elif entry.get("status") == "miss" and all(key):
                    self._counts[key] += 1

    def _append(self, payload: Dict[str, Any]) -> None:
        self.persist_path.parent.mkdir(parents=True, exist_ok=True)
        with self.persist_path.open("a", encoding="utf-8") as f:
            f.write(json.dumps(payload) + "\n")

    def log_miss(self, question: str, ticker: Optional[str], filing_year: Optional[str], form_type: Optional[str]) -> None:
        t = str(ticker or "").upper()
        y = str(filing_year or "")
        f = str(form_type or "10-K").upper()
        key = (t, y, f)
        if not t or not y or key in self._already_ingested:
            return

        self._counts[key] += 1
        self._append(
            {
                "status": "miss",
                "ticker": t,
                "filing_year": y,
                "form_type": f,
                "question": question,
                "timestamp": datetime.now(timezone.utc).isoformat(),
            }
        )
        print(f"  [DriftTracker] Miss #{self._counts[key]} for ({t}, {y}, {f})")

    def get_pending_scrapes(self) -> List[Tuple[str, str, str]]:
        return [
            (t, y, f)
            for (t, y, f), count in self._counts.items()
            if count >= self.TRIGGER_THRESHOLD and (t, y, f) not in self._already_ingested
        ]

    def mark_ingested(self, ticker: str, filing_year: str, form_type: str) -> None:
        key = (str(ticker).upper(), str(filing_year), str(form_type).upper())
        self._already_ingested.add(key)
        self._append(
            {
                "status": "ingested",
                "ticker": key[0],
                "filing_year": key[1],
                "form_type": key[2],
                "timestamp": datetime.now(timezone.utc).isoformat(),
            }
        )


DRIFT_LOG_PATH = resolve_path_from_env(CONFIG["sec_chunks_path"]).parent.parent / "drift_log.jsonl"
drift_tracker = DriftTracker(DRIFT_LOG_PATH)
print(f"DriftTracker ready | persist path: {drift_tracker.persist_path}")


def _strip_html(html: str) -> str:
    text = re.sub(r"<[^>]+>", " ", html)
    for ent, repl in [("&nbsp;", " "), ("&amp;", "&"), ("&lt;", "<"), ("&gt;", ">"), ("&#160;", " ")]:
        text = text.replace(ent, repl)
    return re.sub(r"\s{3,}", "\n\n", text).strip()


def fetch_edgar_filings_list(ticker: str, form_type: str = "10-K") -> List[Dict[str, Any]]:
    cik = TICKER_TO_CIK.get(ticker.upper())
    if not cik:
        raise ValueError(f"Unknown ticker: {ticker}")

    resp = requests.get(
        f"https://data.sec.gov/submissions/CIK{cik}.json",
        headers=EDGAR_HEADERS,
        timeout=30,
    )
    resp.raise_for_status()
    filings = resp.json().get("filings", {}).get("recent", {})

    out: List[Dict[str, Any]] = []
    forms = filings.get("form", [])
    for i, form in enumerate(forms):
        if form != form_type:
            continue
        out.append(
            {
                "accessionNumber": filings.get("accessionNumber", [""])[i],
                "filingDate": filings.get("filingDate", [""])[i],
                "form": form,
                "primaryDocument": (filings.get("primaryDocument") or [None])[i],
            }
        )
    return out


def fetch_filing_text(ticker: str, accession_number: str, primary_doc: Optional[str]) -> str:
    cik = TICKER_TO_CIK[ticker.upper()].lstrip("0")
    acc_no_dashes = accession_number.replace("-", "")
    base = f"https://www.sec.gov/Archives/edgar/data/{cik}/{acc_no_dashes}"

    if primary_doc:
        resp = requests.get(f"{base}/{primary_doc}", headers=EDGAR_HEADERS, timeout=60)
        if resp.ok:
            return _strip_html(resp.text)

    index_url = f"{base}/{accession_number}-index.htm"
    resp = requests.get(index_url, headers=EDGAR_HEADERS, timeout=30)
    links = re.findall(r'href="(/Archives[^"]+\.htm)"', resp.text, re.IGNORECASE)
    if links:
        resp2 = requests.get(f"https://www.sec.gov{links[0]}", headers=EDGAR_HEADERS, timeout=60)
        if resp2.ok:
            return _strip_html(resp2.text)

    raise RuntimeError(f"Could not fetch filing for {ticker} {accession_number}")


def scrape_filing(ticker: str, filing_year: str, form_type: str = "10-K") -> Dict[str, Any]:
    filings = fetch_edgar_filings_list(ticker, form_type)
    match = next((f for f in filings if f["filingDate"].startswith(str(filing_year))), None)
    if match is None:
        match = next((f for f in filings if f["filingDate"].startswith(str(int(filing_year) + 1))), None)
    if match is None:
        raise RuntimeError(f"No {form_type} found for {ticker} FY{filing_year} on EDGAR")

    text = fetch_filing_text(ticker, match["accessionNumber"], match.get("primaryDocument"))
    print(
        f"  [EDGAR] Downloaded {ticker} {form_type} {filing_year} "
        f"({len(text):,} chars, filed {match['filingDate']})"
    )
    return {
        "ticker": ticker.upper(),
        "filing_year": str(filing_year),
        "form_type": form_type.upper(),
        "filing_date": match["filingDate"],
        "accession_number": match["accessionNumber"],
        "text": text,
    }


DRIFT_CHUNK_WORDS = 800
DRIFT_CHUNK_OVERLAP = 100


def chunk_filing(filing: Dict[str, Any]) -> List[Dict[str, Any]]:
    words = filing["text"].split()
    company = COMPANY_NAMES_MAP.get(filing["ticker"], filing["ticker"])

    chunks: List[Dict[str, Any]] = []
    start = 0
    idx = 0
    while start < len(words):
        end = min(start + DRIFT_CHUNK_WORDS, len(words))
        text = " ".join(words[start:end])
        cid = f"{filing['ticker']}_{filing['form_type']}_{filing['filing_year']}_{idx:04d}_drift"
        contextual = (
            f"Company: {company} ({filing['ticker']})\n"
            f"Filing: {filing['form_type']} | Date: {filing['filing_date']} | Year: {filing['filing_year']}\n"
            f"Section: auto-chunked\n"
            f"Content: {text}"
        )
        chunks.append(
            {
                "chunk_id": cid,
                "ticker": filing["ticker"],
                "company_name": company,
                "form_type": filing["form_type"],
                "filing_year": str(filing["filing_year"]),
                "filing_date": filing["filing_date"],
                "accession_number": filing["accession_number"],
                "section_title": "auto-chunked",
                "chunk_index": idx,
                "text": text,
                "contextual_text": contextual,
                "word_count": end - start,
                "source": "drift_ingested",
                "flag_low_value_combined": False,
            }
        )
        start += DRIFT_CHUNK_WORDS - DRIFT_CHUNK_OVERLAP
        idx += 1

    print(f"  [Chunker] {len(chunks)} chunks from {filing['ticker']} {filing['form_type']} {filing['filing_year']}")
    return chunks


def ingest_to_chroma(chunks: List[Dict[str, Any]], corpus_index: CorpusIndex) -> int:
    collection = corpus_index.collection
    ids = [c["chunk_id"] for c in chunks]
    existing = set(collection.get(ids=ids).get("ids", []))
    new_chunks = [c for c in chunks if c["chunk_id"] not in existing]
    if not new_chunks:
        print("  [Ingestor] All chunks already present; skipping.")
        return 0

    texts = [c["contextual_text"] for c in new_chunks]
    embs = embed_model.encode(texts, batch_size=32, show_progress_bar=False).tolist()
    metas = [{k: v for k, v in c.items() if k not in ("text", "contextual_text")} for c in new_chunks]

    collection.upsert(
        ids=[c["chunk_id"] for c in new_chunks],
        embeddings=embs,
        documents=texts,
        metadatas=metas,
    )
    print(f"  [Ingestor] Added {len(new_chunks)} chunks to ChromaDB.")
    return len(new_chunks)


def update_corpus_index(corpus_index: CorpusIndex, new_chunks: List[Dict[str, Any]]) -> None:
    base_len = len(corpus_index.df)
    rows: List[Dict[str, Any]] = []

    for i, c in enumerate(new_chunks):
        doc_name = f"{c['ticker']}_{c['form_type']}_{str(c['filing_date'])[:10]}"
        contextual = c["contextual_text"]
        rows.append(
            {
                "row_idx": base_len + i,
                "chunk_id_str": c["chunk_id"],
                "ticker": c["ticker"],
                "company": c["company_name"],
                "doc_name": doc_name,
                "filing_year": int(c["filing_year"]),
                "form_type": c["form_type"],
                "page_num": int(c["chunk_index"]),
                "raw_chunk": c["text"],
                "contextual_chunk": contextual,
                "bm25_tokens": contextual.lower().split(),
            }
        )

    if rows:
        corpus_index.df = pd.concat([corpus_index.df, pd.DataFrame(rows)], ignore_index=True)
        corpus_index.bm25 = BM25Okapi(corpus_index.df["bm25_tokens"].tolist())

    sec_chunks_out = resolve_path_from_env(CONFIG["sec_chunks_path"])
    sec_chunks_out.parent.mkdir(parents=True, exist_ok=True)
    with sec_chunks_out.open("a", encoding="utf-8") as f:
        for c in new_chunks:
            f.write(json.dumps(c) + "\n")

    print(f"  [BM25] Index rebuilt: {len(corpus_index.df):,} total chunks.")


class DriftDetectEvent(Event):
    state: AgentState


class ScrapeIngestEvent(Event):
    state: AgentState


class JudgeEvent(Event):
    state: AgentState


class SecAgenticRAGWorkflow(Workflow):
    @step
    async def query_planning(self, _ctx: Context, ev: StartEvent) -> PlanEvent:
        question = ev.get("question")
        index = ev.get("index")

        if not question or index is None:
            raise ValueError("Workflow requires question and index")

        state: AgentState = {
            "question": question,
            "index": index,
            "context_retries": 0,
            "repair_used": False,
            "repair_retrieval_count": 0,
            "is_repair_retrieval": False,
            "retrieval_sanity_failed": False,
            "sub_queries": [],
            "needs_decomposition": False,
            "drift_pending_scrapes": [],
            "drift_triggered": False,
            "ingestion_just_ran": False,
            "golden_answer": ev.get("golden_answer"),
            "judge_score": None,
            "judge_claims_covered": None,
            "judge_reason": None,
        }

        result = invoke_structured(
            planner_llm,
            PlannerOutput,
            PLANNER_SYSTEM,
            f"Question:\n{question}",
            PlannerOutput(needs_decomposition=False, sub_queries=[SubQuery(query=question)]),
        )

        sub_queries: List[Dict[str, Any]] = []
        for sq in result.sub_queries:
            sq_dict = sq.model_dump()
            if not sq_dict.get("query"):
                parts = [question]
                if sq_dict.get("ticker"):
                    parts.append(f"ticker {sq_dict['ticker']}")
                if sq_dict.get("filing_year"):
                    parts.append(f"filing year {sq_dict['filing_year']}")
                if sq_dict.get("form_type"):
                    parts.append(f"form {sq_dict['form_type']}")
                sq_dict["query"] = " | ".join(parts)
            sq_dict["query"] = cleanup_planner_query(
                sq_dict.get("query", ""),
                ticker=sq_dict.get("ticker"),
                filing_year=sq_dict.get("filing_year"),
                form_type=sq_dict.get("form_type"),
            )
            sub_queries.append(sq_dict)

        if not sub_queries:
            sub_queries = [
                {
                    "query": cleanup_planner_query(question),
                    "ticker": None,
                    "filing_year": None,
                    "form_type": None,
                }
            ]

        state["sub_queries"] = sub_queries
        state["rewritten_query"] = sub_queries[0]["query"]
        state["needs_decomposition"] = result.needs_decomposition

        print(
            f"  [Planner] {'decomposed' if state['needs_decomposition'] else 'single'} | "
            f"{len(sub_queries)} sub-quer{'ies' if len(sub_queries) > 1 else 'y'}"
        )

        if state["needs_decomposition"]:
            for sq in sub_queries:
                print(
                    f"    -> {sq['query']}  "
                    f"(ticker={sq.get('ticker')}, year={sq.get('filing_year')}, form={sq.get('form_type')})"
                )

        return PlanEvent(state=state)

    @step
    async def hybrid_retriever(
        self,
        _ctx: Context,
        ev: PlanEvent | RetrieveEvent | RepairRetrieveEvent,
    ) -> RetrievedEvent:
        state = ev.state
        sub_queries = state.get("sub_queries", [])

        if isinstance(ev, RepairRetrieveEvent):
            state["is_repair_retrieval"] = True

        if len(sub_queries) <= 1:
            q = state.get("rewritten_query", state["question"])
            sq = sub_queries[0] if sub_queries else {}
            active_index = state.get("index") or globals().get("global_index")
            if active_index is None:
                raise KeyError("index")
            retrieved = active_index.hybrid_search(
                query=q,
                # shared retriever uses models from its own index state,
                bm25_top_k=CONFIG["bm25_top_k"],
                dense_top_k=CONFIG["dense_top_k"],
                rerank_top_k=CONFIG["rerank_top_k"],
                ticker=sq.get("ticker"),
                filing_year=sq.get("filing_year"),
                form_type=sq.get("form_type"),
            )
        else:
            per_k = CONFIG["decomposition_top_k_per_subquery"]
            merged: Dict[str, RetrievedChunk] = {}
            for sq in sub_queries:
                active_index = state.get("index") or globals().get("global_index")
                if active_index is None:
                    raise KeyError("index")
                chunks = active_index.hybrid_search(
                    query=sq["query"],
                    # shared retriever uses models from its own index state,
                    bm25_top_k=CONFIG["bm25_top_k"],
                    dense_top_k=CONFIG["dense_top_k"],
                    rerank_top_k=per_k,
                    ticker=sq.get("ticker"),
                    filing_year=sq.get("filing_year"),
                    form_type=sq.get("form_type"),
                )
                for chunk in chunks:
                    if chunk.chunk_id not in merged or chunk.score > merged[chunk.chunk_id].score:
                        merged[chunk.chunk_id] = chunk

            retrieved = sorted(merged.values(), key=lambda x: x.score, reverse=True)[: CONFIG["rerank_top_k"]]

        state["retrieved_chunks"] = retrieved
        state["retrieved_doc_names"] = extract_retrieved_doc_names(retrieved)
        state["retrieval_sanity_failed"] = fails_retrieval_sanity_check(
            state["question"],
            sub_queries,
            retrieved,
        )

        return RetrievedEvent(state=state)

    @step
    async def context_evaluator(self, _ctx: Context, ev: RetrievedEvent) -> GenerateEvent | RetrieveEvent:
        state = ev.state

        if state.get("retrieval_sanity_failed", False):
            relevant = False
        else:
            context = format_retrieved_context(
                state.get("retrieved_chunks", []),
                max_chunks=CONFIG["control_max_context_chunks"],
                max_chars=CONFIG["control_max_context_chars"],
            )
            judged = invoke_structured(
                context_eval_llm,
                ContextEvalOutput,
                CONTEXT_EVAL_SYSTEM,
                f"Question:\n{state['question']}\n\nContext:\n{context}",
                ContextEvalOutput(relevant=True, reason="fallback"),
            )
            relevant = judged.relevant

        state["context_eval_relevant"] = relevant

        if relevant:
            return GenerateEvent(state=state)

        retries = state.get("context_retries", 0)
        if retries < CONFIG["max_context_retries"]:
            state["context_retries"] = retries + 1
            return RetrieveEvent(state=state)

        return GenerateEvent(state=state)

    @step
    async def generator(self, _ctx: Context, ev: GenerateEvent) -> CritiqueEvent:
        state = ev.state
        context = format_retrieved_context(
            state.get("retrieved_chunks", []),
            max_chunks=CONFIG["generator_max_context_chunks"],
            max_chars=CONFIG["generator_max_context_chars"],
        )

        answer = invoke_text(
            generator_llm,
            GENERATOR_SYSTEM,
            f"Question:\n{state['question']}\n\nRetrieved context:\n{context}",
            fallback_text="",
        ).strip()

        state["generated_answer"] = answer
        state["final_answer"] = answer
        return CritiqueEvent(state=state)

    @step
    async def critic(self, _ctx: Context, ev: CritiqueEvent) -> JudgeEvent | RepairEvent | DriftDetectEvent:
        state = ev.state

        if state.get("is_repair_retrieval", False):
            return JudgeEvent(state=state)

        context = format_retrieved_context(
            state.get("retrieved_chunks", []),
            max_chunks=CONFIG["control_max_context_chunks"],
            max_chars=CONFIG["control_max_context_chars"],
        )

        result = invoke_structured(
            critic_llm,
            CriticOutput,
            CRITIC_SYSTEM,
            (
                f"Question:\n{state['question']}\n\n"
                f"Context:\n{context}\n\n"
                f"Answer:\n{state.get('generated_answer', '')}"
            ),
            CriticOutput(decision="accept", reason="fallback"),
        )

        state["critic_decision"] = result.decision
        state["critic_reason"] = result.reason

        if result.decision == "repair":
            return RepairEvent(state=state)

        if result.decision == "insufficient_data":
            state["final_answer"] = (
                "Insufficient data: The required information could not be found in the "
                f"retrieved SEC filings. ({result.reason})"
            )
            if CONFIG.get("enable_drift_and_scrape", False):
                return DriftDetectEvent(state=state)
            return JudgeEvent(state=state)

        return JudgeEvent(state=state)

    @step
    async def repair(self, _ctx: Context, ev: RepairEvent) -> JudgeEvent | RepairRetrieveEvent:
        state = ev.state
        context = format_retrieved_context(
            state.get("retrieved_chunks", []),
            max_chunks=CONFIG["control_max_context_chunks"],
            max_chars=CONFIG["control_max_context_chars"],
        )

        result = invoke_structured(
            repair_llm,
            RepairOutput,
            REPAIR_SYSTEM,
            (
                f"Question:\n{state['question']}\n\n"
                f"Context:\n{context}\n\n"
                f"Answer:\n{state.get('generated_answer', '')}\n\n"
                f"Critique:\n{state.get('critic_reason', 'Needs repair')}"
            ),
            RepairOutput(
                decision="accept",
                revised_answer=state.get("generated_answer", ""),
                needs_new_retrieval=False,
                reason="fallback",
            ),
        )

        state["repair_used"] = True
        state["repair_decision"] = result.decision
        state["repair_reason"] = result.reason
        state["needs_new_retrieval"] = result.needs_new_retrieval
        state["final_answer"] = (result.revised_answer or "").strip()

        if result.needs_new_retrieval:
            count = state.get("repair_retrieval_count", 0) + 1
            state["repair_retrieval_count"] = count
            if count <= CONFIG["max_repair_retrievals"]:
                state["is_repair_retrieval"] = True
                return RepairRetrieveEvent(state=state)

        return JudgeEvent(state=state)

    @step
    async def drift_detector(self, _ctx: Context, ev: DriftDetectEvent) -> JudgeEvent | ScrapeIngestEvent:
        state = ev.state
        sub_queries = state.get("sub_queries", [])

        if not CONFIG.get("enable_drift_and_scrape", False):
            return JudgeEvent(state=state)

        ticker = None
        filing_year = None
        form_type = "10-K"
        if sub_queries:
            sq = sub_queries[0]
            ticker = sq.get("ticker")
            filing_year = str(sq.get("filing_year")) if sq.get("filing_year") else None
            form_type = str(sq.get("form_type") or "10-K")

        drift_tracker.log_miss(
            question=state["question"],
            ticker=ticker,
            filing_year=filing_year,
            form_type=form_type,
        )

        pending = drift_tracker.get_pending_scrapes()
        state["drift_pending_scrapes"] = pending
        state["drift_triggered"] = len(pending) > 0
        state["ingestion_just_ran"] = False

        print(f"  [DriftDetector] Pending scrapes: {pending}")

        if state["drift_triggered"] and pending:
            return ScrapeIngestEvent(state=state)

        return JudgeEvent(state=state)

    @step
    async def scrape_and_ingest(self, _ctx: Context, ev: ScrapeIngestEvent) -> JudgeEvent | RetrieveEvent:
        state = ev.state
        corpus_index = state["index"]
        ingested_any = False

        for ticker, filing_year, form_type in state.get("drift_pending_scrapes", []):
            try:
                filing = scrape_filing(ticker, str(filing_year), form_type)
                chunks = chunk_filing(filing)
                added = ingest_to_chroma(chunks, corpus_index)
                if added > 0:
                    update_corpus_index(corpus_index, chunks)
                    ingested_any = True
                drift_tracker.mark_ingested(ticker, str(filing_year), form_type)
            except Exception as e:
                print(f"  [ScrapeIngest] Failed for {ticker} {filing_year}: {e}")

        state["ingestion_just_ran"] = ingested_any
        state["context_retries"] = 0
        state["critic_decision"] = None
        state["is_repair_retrieval"] = False

        if ingested_any:
            return RetrieveEvent(state=state)

        return JudgeEvent(state=state)

    @step
    async def judge_final_answer(self, _ctx: Context, ev: JudgeEvent) -> StopEvent:
        state = ev.state
        reference = str(state.get("golden_answer") or "").strip()
        candidate = str(state.get("final_answer") or "").strip()

        if reference:
            score, claims, reason = llm_judge_correctness(
                state["question"],
                reference,
                candidate,
            )
            state["judge_score"] = score
            state["judge_claims_covered"] = claims
            state["judge_reason"] = reason
        else:
            state["judge_score"] = None
            state["judge_claims_covered"] = None
            state["judge_reason"] = "Skipped: no golden_answer supplied."

        return StopEvent(result=state)


print("SecAgenticRAGWorkflow updated (judge integrated; drift toggle enabled).")

DriftTracker ready | persist path: C:\Users\wenxu\GenAI-with-LLMs\sec_rag_team_share\sec_data\drift_log.jsonl
SecAgenticRAGWorkflow updated (judge integrated; drift toggle enabled).


In [8]:
# Compatibility shims used by downstream parity-alignment cells.
if 'RateLimiter' not in globals():
    class RateLimiter:
        def __init__(self, max_rpm: int):
            self.max_rpm = max_rpm
            self.window = 60.0
            self._timestamps = deque()
            self._lock = threading.Lock()

        def wait(self):
            with self._lock:
                now = time.time()
                while self._timestamps and now - self._timestamps[0] >= self.window:
                    self._timestamps.popleft()
                if len(self._timestamps) >= self.max_rpm:
                    sleep_for = self.window - (now - self._timestamps[0])
                    if sleep_for > 0:
                        time.sleep(sleep_for)
                self._timestamps.append(time.time())

if 'resolve_role_temperature' not in globals():
    def resolve_role_temperature(role: str, task_name: Optional[str] = None) -> float:
        if task_name:
            tk = f"temperature_{task_name}"
            if tk in CONFIG:
                return float(CONFIG[tk])
        rk = f"temperature_{role}"
        if rk in CONFIG:
            return float(CONFIG[rk])
        return float(CONFIG.get('temperature_generator', 0.2))

print('Compatibility shims ready (RateLimiter, resolve_role_temperature).')

Compatibility shims ready (RateLimiter, resolve_role_temperature).


In [9]:
# ---------------------------------------------------------------------------
# Parity alignment patch: bring LlamaIndex workflow behavior closer to
# LangGraph SEC v3 while preserving path portability logic.
# ---------------------------------------------------------------------------

# 1) Configuration alignment (do not alter path portability keys)
CONFIG["execution_profile"] = "dev"
CONFIG["use_pilot"] = True
CONFIG["pilot_n_questions"] = 10
CONFIG["full_n_questions"] = 80
CONFIG["use_llm_judge"] = False
CONFIG["use_few_shot_examples"] = True
CONFIG["pilot_judge_sample_n"] = 1
CONFIG["full_judge_sample_n"] = 2
CONFIG["judge_max_context_chunks"] = 3
CONFIG["judge_max_context_chars"] = 4000

# Keep existing drift toggle semantics in drift_detector,
# but update critic routing to always pass through drift_detector.
CONFIG["enable_drift_and_scrape"] = bool(
    CONFIG.get("enable_drift_and_scrape", CONFIG.get("ENABLE_DRIFT_AND_SCRAPE", False))
)

# LangGraph-style provider aware RPM.
CONFIG["max_rpm"] = 28 if CONFIG["provider"] == "groq" else 10
CONFIG["judge_sample_n"] = (
    CONFIG["pilot_judge_sample_n"] if CONFIG["use_pilot"] else CONFIG["full_judge_sample_n"]
) if CONFIG["use_llm_judge"] else 0

# LangGraph-style fallback model lists.
if CONFIG["provider"] == "groq":
    CONFIG.setdefault("groq_fallback_agent_models", ["llama-3.1-8b-instant", "qwen/qwen3-32b"])
    CONFIG.setdefault("groq_fallback_generator_models", ["llama-3.1-8b-instant", "qwen/qwen3-32b"])
    CONFIG.setdefault("groq_fallback_judge_models", ["llama-3.1-8b-instant", "qwen/qwen3-32b"])
if CONFIG["provider"] == "gemini":
    CONFIG.setdefault("gemini_fallback_agent_models", [])
    CONFIG.setdefault("gemini_fallback_generator_models", [])
    CONFIG.setdefault("gemini_fallback_judge_models", [])

# Rebuild limiter after max_rpm update.
rate_limiter = RateLimiter(CONFIG["max_rpm"])


def make_llm_from_model(model_name: str, temperature: float):
    if CONFIG["provider"] == "groq":
        api_key = os.getenv("GROQ_API_KEY", "")
        if not api_key:
            raise ValueError("GROQ_API_KEY is missing in .env")
        return LIGroq(model=model_name, api_key=api_key, temperature=temperature)

    if CONFIG["provider"] == "gemini":
        api_key = os.getenv("GOOGLE_API_KEY", "")
        if not api_key:
            raise ValueError("GOOGLE_API_KEY is missing in .env")
        return LIGemini(model=model_name, api_key=api_key, temperature=temperature)

    raise ValueError(f"Unsupported provider: {CONFIG['provider']}")


def make_llm(role: str, task_name: Optional[str] = None):
    model_name = resolve_model_name(role)
    temp = resolve_role_temperature(role, task_name)
    return make_llm_from_model(model_name, temp)


def resolve_fallback_model_names(role: str) -> List[str]:
    provider = CONFIG["provider"]
    primary = resolve_model_name(role)
    fallbacks = CONFIG.get(f"{provider}_fallback_{role}_models", [])
    ordered = [primary] + list(fallbacks)
    return list(dict.fromkeys([m for m in ordered if m]))


def get_llm_candidates(role: str, task_name: Optional[str] = None):
    temperature = resolve_role_temperature(role, task_name)
    return [
        (model_name, make_llm_from_model(model_name, temperature))
        for model_name in resolve_fallback_model_names(role)
    ]


_preferred_models_by_task: Dict[str, str] = {}


def order_model_candidates(role: str, task_name: Optional[str] = None):
    candidates = get_llm_candidates(role, task_name)
    preference_key = task_name or role
    preferred = _preferred_models_by_task.get(preference_key)
    if not preferred:
        return candidates
    names = [name for name, _ in candidates]
    if preferred not in names:
        return candidates
    return sorted(candidates, key=lambda item: item[0] != preferred)


def is_rate_limit_error(exc: Exception) -> bool:
    msg = str(exc).lower()
    return "rate limit" in msg or "rate_limit" in msg or "429" in msg


def _infer_role_task_from_llm(llm_obj) -> Optional[tuple[str, str]]:
    if llm_obj is planner_llm:
        return ("agent", "planner")
    if llm_obj is context_eval_llm:
        return ("agent", "context_eval")
    if llm_obj is generator_llm:
        return ("generator", "generator")
    if llm_obj is critic_llm:
        return ("agent", "critic")
    if llm_obj is repair_llm:
        return ("agent", "repair")
    if llm_obj is judge_llm:
        return ("judge", "judge")
    return None


def _question_tokens(text: str) -> set[str]:
    return {tok for tok in re.findall(r"[a-z0-9]+", str(text).lower()) if len(tok) > 2}


def select_few_shot_examples(question: str, max_examples: int = 2) -> str:
    if not CONFIG.get("use_few_shot_examples", False):
        return ""
    if "train_df" not in globals():
        return ""
    if train_df is None or len(train_df) == 0:
        return ""

    q_tokens = _question_tokens(question)
    scored: List[tuple[int, str, str]] = []
    for _, row in train_df.iterrows():
        ex_q = str(row.get("question", "")).strip()
        ex_a = str(row.get("reference_answer", "")).strip()
        if not ex_q or not ex_a:
            continue
        overlap = len(q_tokens & _question_tokens(ex_q))
        scored.append((overlap, ex_q, ex_a))

    if not scored:
        return ""

    scored.sort(key=lambda x: x[0], reverse=True)
    top = scored[:max_examples]
    lines: List[str] = []
    for i, (_, ex_q, ex_a) in enumerate(top, start=1):
        lines.append(f"Example {i} Question: {ex_q}")
        lines.append(f"Example {i} Answer: {ex_a}")
    return "\n".join(lines)


def _extract_question_from_generator_user_block(user: str) -> str:
    m = re.search(r"Question:\n(.*?)\n\nRetrieved context:", user, flags=re.DOTALL)
    if not m:
        return ""
    return m.group(1).strip()


def invoke_structured(
    llm_or_role,
    schema,
    system: str,
    user: str,
    fallback,
    task_name: Optional[str] = None,
):
    prompt = f"System:\n{system}\n\nUser:\n{user}"

    # Preferred path: role-based fallback strategy.
    if isinstance(llm_or_role, str):
        role = llm_or_role
        candidates = order_model_candidates(role, task_name)
        preference_key = task_name or role
        max_attempts = int(CONFIG["llm_max_retries"])

        for model_idx, (model_name, llm) in enumerate(candidates):
            for attempt in range(max_attempts):
                try:
                    raw = llm_complete(llm, prompt)
                    data = json.loads(_strip_code_fence(raw))
                    parsed = schema.model_validate(data)
                    _preferred_models_by_task[preference_key] = model_name
                    return parsed
                except Exception as e:
                    print(
                        f"  [WARN] {schema.__name__} attempt {attempt + 1}/{max_attempts} "
                        f"on {model_name} failed: {e}"
                    )
                    should_switch = is_rate_limit_error(e) and model_idx < len(candidates) - 1
                    if should_switch:
                        break
                    if attempt < max_attempts - 1:
                        time.sleep(CONFIG["llm_retry_base_delay_sec"] * (2 ** attempt))
        return fallback

    # Backward compatibility path: infer role/task from llm object if possible.
    inferred = _infer_role_task_from_llm(llm_or_role)
    if inferred is not None:
        role, inferred_task = inferred
        return invoke_structured(
            role,
            schema,
            system,
            user,
            fallback,
            task_name=task_name or inferred_task,
        )

    # Last resort legacy behavior for unknown llm objects.
    for attempt in range(int(CONFIG["llm_max_retries"])):
        try:
            raw = llm_complete(llm_or_role, prompt)
            data = json.loads(_strip_code_fence(raw))
            return schema.model_validate(data)
        except Exception:
            if attempt == int(CONFIG["llm_max_retries"]) - 1:
                return fallback
            time.sleep(CONFIG["llm_retry_base_delay_sec"] * (2 ** attempt))
    return fallback


def invoke_text(
    llm_or_role,
    system: str,
    user: str,
    fallback_text: str = "",
    task_name: Optional[str] = None,
) -> str:
    # Preferred path: role-based fallback strategy.
    if isinstance(llm_or_role, str):
        role = llm_or_role
        candidates = order_model_candidates(role, task_name)
        preference_key = task_name or role
        max_attempts = int(CONFIG["llm_max_retries"])

        effective_user = user
        if (task_name or "") == "generator":
            question = _extract_question_from_generator_user_block(user)
            few_shot = select_few_shot_examples(question)
            if few_shot:
                effective_user = f"Few-shot examples:\n{few_shot}\n\n{user}"

        prompt = f"System:\n{system}\n\nUser:\n{effective_user}"

        for model_idx, (model_name, llm) in enumerate(candidates):
            for attempt in range(max_attempts):
                try:
                    result = llm_complete(llm, prompt)
                    _preferred_models_by_task[preference_key] = model_name
                    return result
                except Exception as e:
                    print(
                        f"  [WARN] text invoke attempt {attempt + 1}/{max_attempts} "
                        f"on {model_name} failed: {e}"
                    )
                    should_switch = is_rate_limit_error(e) and model_idx < len(candidates) - 1
                    if should_switch:
                        break
                    if attempt < max_attempts - 1:
                        time.sleep(CONFIG["llm_retry_base_delay_sec"] * (2 ** attempt))
        return fallback_text

    inferred = _infer_role_task_from_llm(llm_or_role)
    if inferred is not None:
        role, inferred_task = inferred
        return invoke_text(
            role,
            system,
            user,
            fallback_text=fallback_text,
            task_name=task_name or inferred_task,
        )

    prompt = f"System:\n{system}\n\nUser:\n{user}"
    try:
        return llm_complete(llm_or_role, prompt)
    except Exception:
        return fallback_text


# Recreate role llm handles after execution profile update.
planner_llm = make_llm("agent", task_name="planner")
context_eval_llm = make_llm("agent", task_name="context_eval")
generator_llm = make_llm("generator", task_name="generator")
critic_llm = make_llm("agent", task_name="critic")
repair_llm = make_llm("agent", task_name="repair")
judge_llm = make_llm("judge", task_name="judge")

JUDGE_CORRECTNESS_WITH_CONTEXT_SYSTEM = (
    "You are a strict evaluator. Score correctness between 0.0 and 1.0 and claims_covered between 0.0 and 1.0. "
    "Use the reference answer as the source of truth and use provided retrieved context for grounding checks. "
    "Return JSON with keys: score, claims_covered, reason."
)


# 2) Workflow behavior alignment
#    - critic routes insufficient_data to drift detector (like LangGraph graph path)
#    - judge uses explicit judge context caps
_SecAgenticRAGWorkflowBase = SecAgenticRAGWorkflow


class SecAgenticRAGWorkflow(_SecAgenticRAGWorkflowBase):
    @step
    async def critic(self, _ctx: Context, ev: CritiqueEvent) -> JudgeEvent | RepairEvent | DriftDetectEvent:
        state = ev.state

        if state.get("is_repair_retrieval", False):
            return JudgeEvent(state=state)

        context = format_retrieved_context(
            state.get("retrieved_chunks", []),
            max_chunks=CONFIG["control_max_context_chunks"],
            max_chars=CONFIG["control_max_context_chars"],
        )

        result = invoke_structured(
            "agent",
            CriticOutput,
            CRITIC_SYSTEM,
            (
                f"Question:\n{state['question']}\n\n"
                f"Context:\n{context}\n\n"
                f"Answer:\n{state.get('generated_answer', '')}"
            ),
            CriticOutput(decision="accept", reason="fallback"),
            task_name="critic",
        )

        state["critic_decision"] = result.decision
        state["critic_reason"] = result.reason

        if result.decision == "repair":
            return RepairEvent(state=state)

        if result.decision == "insufficient_data":
            state["final_answer"] = (
                "Insufficient data: The required information could not be found in the "
                f"retrieved SEC filings. ({result.reason})"
            )
            # Always route through drift detector for parity with LangGraph routing.
            return DriftDetectEvent(state=state)

        return JudgeEvent(state=state)

    @step
    async def judge_final_answer(self, _ctx: Context, ev: JudgeEvent) -> StopEvent:
        state = ev.state
        reference = str(state.get("golden_answer") or "").strip()
        candidate = str(state.get("final_answer") or "").strip()

        judge_context = format_retrieved_context(
            state.get("retrieved_chunks", []),
            max_chunks=int(CONFIG.get("judge_max_context_chunks", 3)),
            max_chars=int(CONFIG.get("judge_max_context_chars", 4000)),
        )

        if reference:
            judged = invoke_structured(
                "judge",
                JudgeOutput,
                JUDGE_CORRECTNESS_WITH_CONTEXT_SYSTEM,
                (
                    f"Question:\n{state['question']}\n\n"
                    f"Reference answer:\n{reference}\n\n"
                    f"Retrieved context:\n{judge_context}\n\n"
                    f"Candidate answer:\n{candidate}"
                ),
                JudgeOutput(score=0.0, claims_covered=0.0, reason="Judge fallback"),
                task_name="judge",
            )
            state["judge_score"] = max(0.0, min(1.0, float(judged.score)))
            state["judge_claims_covered"] = max(0.0, min(1.0, float(judged.claims_covered)))
            state["judge_reason"] = judged.reason
        else:
            state["judge_score"] = None
            state["judge_claims_covered"] = None
            state["judge_reason"] = "Skipped: no golden_answer supplied."

        return StopEvent(result=state)


print("Parity alignment patch applied (config + fallback + few-shot + drift/judge behavior).")

C:\Users\wenxu\AppData\Local\Temp\ipykernel_4636\3103419830.py:55: DeprecationWarning: Call to deprecated class Gemini. (Should use `llama-index-llms-google-genai` instead, using Google's latest unified SDK. See: https://docs.llamaindex.ai/en/stable/examples/llm/google_genai/This package will no longer be supported after version 0.6.2) -- Deprecated since version 0.6.2.
  return LIGemini(model=model_name, api_key=api_key, temperature=temperature)


C:\Users\wenxu\AppData\Local\Temp\ipykernel_4636\3103419830.py:55: DeprecationWarning: Call to deprecated class Gemini. (Should use `llama-index-llms-google-genai` instead, using Google's latest unified SDK. See: https://docs.llamaindex.ai/en/stable/examples/llm/google_genai/This package will no longer be supported after version 0.6.2) -- Deprecated since version 0.6.2.
  return LIGemini(model=model_name, api_key=api_key, temperature=temperature)


C:\Users\wenxu\AppData\Local\Temp\ipykernel_4636\3103419830.py:55: DeprecationWarning: Call to deprecated class Gemini. (Should use `llama-index-llms-google-genai` instead, using Google's latest unified SDK. See: https://docs.llamaindex.ai/en/stable/examples/llm/google_genai/This package will no longer be supported after version 0.6.2) -- Deprecated since version 0.6.2.
  return LIGemini(model=model_name, api_key=api_key, temperature=temperature)
C:\Users\wenxu\AppData\Local\Temp\ipykernel_4636\3103419830.py:55: DeprecationWarning: Call to deprecated class Gemini. (Should use `llama-index-llms-google-genai` instead, using Google's latest unified SDK. See: https://docs.llamaindex.ai/en/stable/examples/llm/google_genai/This package will no longer be supported after version 0.6.2) -- Deprecated since version 0.6.2.
  return LIGemini(model=model_name, api_key=api_key, temperature=temperature)


C:\Users\wenxu\AppData\Local\Temp\ipykernel_4636\3103419830.py:55: DeprecationWarning: Call to deprecated class Gemini. (Should use `llama-index-llms-google-genai` instead, using Google's latest unified SDK. See: https://docs.llamaindex.ai/en/stable/examples/llm/google_genai/This package will no longer be supported after version 0.6.2) -- Deprecated since version 0.6.2.
  return LIGemini(model=model_name, api_key=api_key, temperature=temperature)


C:\Users\wenxu\AppData\Local\Temp\ipykernel_4636\3103419830.py:55: DeprecationWarning: Call to deprecated class Gemini. (Should use `llama-index-llms-google-genai` instead, using Google's latest unified SDK. See: https://docs.llamaindex.ai/en/stable/examples/llm/google_genai/This package will no longer be supported after version 0.6.2) -- Deprecated since version 0.6.2.
  return LIGemini(model=model_name, api_key=api_key, temperature=temperature)


Parity alignment patch applied (config + fallback + few-shot + drift/judge behavior).


In [10]:
# ---------------------------------------------------------------------------
# JSON parsing hardening + judge prompt refinement
# ---------------------------------------------------------------------------


def _strip_code_fence(text: str) -> str:
    """Extract the first JSON object-like block from noisy model output."""
    if text is None:
        return ""
    s = str(text).strip()
    match = re.search(r"\{[\s\S]*\}", s)
    if match:
        return match.group(0).strip()
    return s


def invoke_structured(
    llm_or_role,
    schema,
    system: str,
    user: str,
    fallback,
    task_name: Optional[str] = None,
):
    prompt = f"System:\n{system}\n\nUser:\n{user}"

    # Preferred path: role-based fallback strategy.
    if isinstance(llm_or_role, str):
        role = llm_or_role
        candidates = order_model_candidates(role, task_name)
        preference_key = task_name or role
        max_attempts = int(CONFIG["llm_max_retries"])

        for model_idx, (model_name, llm) in enumerate(candidates):
            for attempt in range(max_attempts):
                try:
                    raw = llm_complete(llm, prompt)
                    cleaned = _strip_code_fence(raw)
                    try:
                        data = json.loads(cleaned)
                    except json.JSONDecodeError as jde:
                        print(
                            f"  [WARN] {schema.__name__} JSON decode failed on {model_name} "
                            f"attempt {attempt + 1}/{max_attempts}: {jde}"
                        )
                        print("  [RAW LLM OUTPUT START]")
                        print(raw)
                        print("  [RAW LLM OUTPUT END]")
                        raise
                    parsed = schema.model_validate(data)
                    _preferred_models_by_task[preference_key] = model_name
                    return parsed
                except Exception as e:
                    print(
                        f"  [WARN] {schema.__name__} attempt {attempt + 1}/{max_attempts} "
                        f"on {model_name} failed: {e}"
                    )
                    should_switch = is_rate_limit_error(e) and model_idx < len(candidates) - 1
                    if should_switch:
                        break
                    if attempt < max_attempts - 1:
                        time.sleep(CONFIG["llm_retry_base_delay_sec"] * (2 ** attempt))
        return fallback

    # Backward compatibility path: infer role/task from llm object if possible.
    inferred = _infer_role_task_from_llm(llm_or_role)
    if inferred is not None:
        role, inferred_task = inferred
        return invoke_structured(
            role,
            schema,
            system,
            user,
            fallback,
            task_name=task_name or inferred_task,
        )

    # Last resort legacy behavior for unknown llm objects.
    for attempt in range(int(CONFIG["llm_max_retries"])):
        try:
            raw = llm_complete(llm_or_role, prompt)
            cleaned = _strip_code_fence(raw)
            try:
                data = json.loads(cleaned)
            except json.JSONDecodeError as jde:
                print(
                    f"  [WARN] {schema.__name__} legacy JSON decode failed "
                    f"attempt {attempt + 1}/{int(CONFIG['llm_max_retries'])}: {jde}"
                )
                print("  [RAW LLM OUTPUT START]")
                print(raw)
                print("  [RAW LLM OUTPUT END]")
                raise
            return schema.model_validate(data)
        except Exception:
            if attempt == int(CONFIG["llm_max_retries"]) - 1:
                return fallback
            time.sleep(CONFIG["llm_retry_base_delay_sec"] * (2 ** attempt))
    return fallback


JUDGE_CORRECTNESS_WITH_CONTEXT_SYSTEM = """
<role>
You are a strict financial QA judge for SEC filing question answering.
</role>

<task>
Evaluate candidate_answer against reference_answer, using retrieved_context only to check grounding.
Return score and claims_covered in [0.0, 1.0].
</task>

<rubric>
- score: overall factual correctness vs reference (numbers, entities, period, directionality).
- claims_covered: fraction of key reference claims covered by candidate.
- reason: concise rationale (1-3 sentences).
</rubric>

<output_format>
Return exactly one JSON object with keys: score, claims_covered, reason.
</output_format>

<hard_constraint>
Do not include any introductory text or summary. Start your response with { and end with }.
</hard_constraint>
""".strip()


def _judge_few_shot_examples() -> str:
    return (
        "Example A (Perfect)\n"
        "Reference: Apple gross margin rose from 44.1% in FY2023 to 46.6% in FY2024.\n"
        "Candidate: Apple's gross margin increased from 44.1% in FY2023 to 46.6% in FY2024.\n"
        "Expected JSON: {\"score\": 1.0, \"claims_covered\": 1.0, \"reason\": \"Matches both values and trend exactly.\"}\n\n"
        "Example B (Poor)\n"
        "Reference: NVIDIA data center revenue increased year over year.\n"
        "Candidate: NVIDIA data center revenue declined sharply year over year.\n"
        "Expected JSON: {\"score\": 0.0, \"claims_covered\": 0.0, \"reason\": \"Contradicts the reference trend.\"}"
    )


_SecAgenticRAGWorkflowJudgeBase = SecAgenticRAGWorkflow


class SecAgenticRAGWorkflow(_SecAgenticRAGWorkflowJudgeBase):
    @step
    async def judge_final_answer(self, _ctx: Context, ev: JudgeEvent) -> StopEvent:
        state = ev.state
        reference = str(state.get("golden_answer") or "").strip()
        candidate = str(state.get("final_answer") or "").strip()

        judge_context = format_retrieved_context(
            state.get("retrieved_chunks", []),
            max_chunks=int(CONFIG.get("judge_max_context_chunks", 3)),
            max_chars=int(CONFIG.get("judge_max_context_chars", 4000)),
        )

        if reference:
            few_shot = _judge_few_shot_examples() if CONFIG.get("use_few_shot_examples", False) else ""
            judged = invoke_structured(
                "judge",
                JudgeOutput,
                JUDGE_CORRECTNESS_WITH_CONTEXT_SYSTEM,
                (
                    (f"Scoring examples:\n{few_shot}\n\n" if few_shot else "")
                    + f"Question:\n{state['question']}\n\n"
                    + f"Reference answer:\n{reference}\n\n"
                    + f"Retrieved context:\n{judge_context}\n\n"
                    + f"Candidate answer:\n{candidate}"
                ),
                JudgeOutput(score=0.0, claims_covered=0.0, reason="Judge fallback"),
                task_name="judge",
            )
            state["judge_score"] = max(0.0, min(1.0, float(judged.score)))
            state["judge_claims_covered"] = max(0.0, min(1.0, float(judged.claims_covered)))
            state["judge_reason"] = judged.reason
        else:
            state["judge_score"] = None
            state["judge_claims_covered"] = None
            state["judge_reason"] = "Skipped: no golden_answer supplied."

        return StopEvent(result=state)


print("JSON parsing and judge prompt hardening patch applied.")

JSON parsing and judge prompt hardening patch applied.


In [11]:
async def run_agentic_rag(question: str, index: CorpusIndex, golden_answer: Optional[str] = None) -> Dict[str, Any]:
    wf = SecAgenticRAGWorkflow(timeout=600, verbose=True)
    result = await wf.run(question=question, index=index, golden_answer=golden_answer)
    return dict(result)


# Quick demo
# question = "How did Apple's gross margin percentage change between fiscal year 2023 and fiscal year 2024?"
# golden_answer = "Apple's gross margin rose from 44.1% in FY2023 to 46.6% in FY2024."
# result = await run_agentic_rag(question, global_index, golden_answer=golden_answer)
#
# print("Question:", question)
# print("Final answer:\n", result.get("final_answer", ""))
# print("Critic decision:", result.get("critic_decision"))
# print("Judge score:", result.get("judge_score"), "| Claims covered:", result.get("judge_claims_covered"))
# print("Repair used:", result.get("repair_used"), "| Repair retrieval count:", result.get("repair_retrieval_count"))

In [12]:
question = "What are the latest risk factors for NVIDIA?"
golden_answer = (
    "NVIDIA's latest risk factors include intense competition, supply chain and manufacturing constraints, "
    "rapid technological change, export controls and geopolitical restrictions, and dependence on key customers/suppliers."
)
result = await run_agentic_rag(question, global_index, golden_answer=golden_answer)

print("Question:", question)
print("Final answer:\n", result.get("final_answer", ""))
print("Critic decision:", result.get("critic_decision"))
print("Judge score:", result.get("judge_score"), "| Claims covered:", result.get("judge_claims_covered"))
print("Judge reason:", result.get("judge_reason"))
print("Repair used:", result.get("repair_used"), "| Repair retrieval count:", result.get("repair_retrieval_count"))
print("Drift triggered:", result.get("drift_triggered"), "| Ingestion ran:", result.get("ingestion_just_ran"))
print("Retrieved docs:", result.get("retrieved_doc_names", []))

Running step query_planning


C:\Users\wenxu\AppData\Local\Temp\ipykernel_4636\3103419830.py:55: DeprecationWarning: Call to deprecated class Gemini. (Should use `llama-index-llms-google-genai` instead, using Google's latest unified SDK. See: https://docs.llamaindex.ai/en/stable/examples/llm/google_genai/This package will no longer be supported after version 0.6.2) -- Deprecated since version 0.6.2.
  return LIGemini(model=model_name, api_key=api_key, temperature=temperature)


  [Planner] single | 1 sub-query
Step query_planning produced event <class '__main__.PlanEvent'>
Running step hybrid_retriever


    [Retrieval] pool=48 (bm25=8 dense=8 adj=32) → rerank top-5
Step hybrid_retriever produced event <class '__main__.RetrievedEvent'>
Running step context_evaluator


C:\Users\wenxu\AppData\Local\Temp\ipykernel_4636\3103419830.py:55: DeprecationWarning: Call to deprecated class Gemini. (Should use `llama-index-llms-google-genai` instead, using Google's latest unified SDK. See: https://docs.llamaindex.ai/en/stable/examples/llm/google_genai/This package will no longer be supported after version 0.6.2) -- Deprecated since version 0.6.2.
  return LIGemini(model=model_name, api_key=api_key, temperature=temperature)


...

Step generator produced event <class '__main__.CritiqueEvent'>
Running step critic


C:\Users\wenxu\AppData\Local\Temp\ipykernel_4636\3103419830.py:55: DeprecationWarning: Call to deprecated class Gemini. (Should use `llama-index-llms-google-genai` instead, using Google's latest unified SDK. See: https://docs.llamaindex.ai/en/stable/examples/llm/google_genai/This package will no longer be supported after version 0.6.2) -- Deprecated since version 0.6.2.
  return LIGemini(model=model_name, api_key=api_key, temperature=temperature)


Step critic produced event <class '__main__.JudgeEvent'>
Running step judge_final_answer


C:\Users\wenxu\AppData\Local\Temp\ipykernel_4636\3103419830.py:55: DeprecationWarning: Call to deprecated class Gemini. (Should use `llama-index-llms-google-genai` instead, using Google's latest unified SDK. See: https://docs.llamaindex.ai/en/stable/examples/llm/google_genai/This package will no longer be supported after version 0.6.2) -- Deprecated since version 0.6.2.
  return LIGemini(model=model_name, api_key=api_key, temperature=temperature)


Step judge_final_answer produced event <class 'workflows.events.StopEvent'>
Question: What are the latest risk factors for NVIDIA?
Final answer:
 The latest risk factors for NVIDIA, as detailed in the filing NVDA_10-K_2026-02-25, include risks related to regulatory, legal, stock, and other matters. Specifically, compliance with existing or future governmental regulations pertaining to IP ownership and infringement, taxes, import and export requirements and tariffs, anti-corruption, business acquisitions, foreign exchange controls and cash repatriation restrictions, data privacy requirements, competition and antitrust, advertising, employment, product regulations, cybersecurity, environmental, health and safety requirements, the responsible use of AI, climate change, cryptocurrency, and consumer laws, could increase costs, impact competitive position, and have a material adverse impact on the business, financial condition, and results of operations.
Critic decision: accept
Judge score: 